# Regression of emittance and Twiss parameters from quadrupole scan data
- Use Gaussian beam



Copyright (C) 2024 Hirokazu Maesaka, RIKEN SPring-8 Center. All rights reserved.

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see <https://www.gnu.org/licenses/>.

## Import modules

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde, gamma, multivariate_normal
import math
import matplotlib.pyplot as plt
%matplotlib inline


## Parameters

In [ ]:
N = 10000   # number of data samples
M = 10000   # number of particles
L = 5.0     # drift length [m]
f0inv = 1.0 # 1/f_0 (reciprocal of minimum focal length of the Q)
f1inv = 0.0 # 1/f_1 (reciprocal of maximum focal length of the Q)
NQ = 21     # number of Q settings

## Generate emittance and Twiss parameters randomly

### Distribution of emittance
- Gamma distribution
- Natural emittance ~ 5 $\mu$ m

In [ ]:
gammarv = gamma(a=5, scale=1.0e-6)
emittance = gammarv.rvs(size=N)
fig, ax = plt.subplots()
y,x,h = ax.hist(emittance*1.0e6, bins=100)
ax.set_xlabel("emittance [$\mu$m rad]")
plt.show()
print(emittance.min(), emittance.mean())

### Distribution of beta
- Gamma distribution
- beta ~ 10 m

In [ ]:
gammarv = gamma(a=10, scale=1.0)
beta = gammarv.rvs(size=N)
fig, ax = plt.subplots()
y,x,h = ax.hist(beta, bins=100)
ax.set_xlabel("beta [m]")
plt.show()
print(beta.min(), beta.mean())

### Distribution of alpha
- uniform distribution within the boundary
- The boundary condition is the existance of a beam waist on the screen.
- $\beta(1/L - 1/f_0) < \alpha < \beta(1/L - 1/f_1)$

In [ ]:
a0, a1 = beta*(1.0/L - f0inv), beta*(1.0/L - f1inv)
urand = np.random.uniform(size=N)
alpha = urand*(a1-a0) + a0
fig, ax = plt.subplots()
y,x,h = ax.hist(alpha, bins=100)
ax.set_xlabel("alpha")
plt.show()

## Generate particles

### Calculate covariance matrix

In [ ]:
gam = (1+alpha**2)/beta
cov = np.rollaxis(emittance * np.array([[beta, -alpha],[-alpha, gam]]), -1)

### Generate particles

In [ ]:
particles = np.empty((N,M,2))
mean = np.zeros(2)
for i in range(N):
    particles[i,:,:] = multivariate_normal.rvs(mean=mean, cov=cov[i,:,:], size=M)

### Check distribution

In [ ]:
for i in range(10):
    eps = np.sqrt(np.linalg.det(cov[i,:,:]))
    a = -cov[i,0,1]/eps
    b = cov[i,0,0]/eps
    g = cov[i,1,1]/eps
    print(i, eps, a, b, g)
    data = particles[i,:,:]
    z = gaussian_kde(data.T)(data.T)
    idx = z.argsort()
    z2 = z[idx]
    fig, ax = plt.subplots(figsize=(3,3))
    ax.scatter(data[idx,0]*1.0e3, data[idx,1]*1.0e3, c=z2, s=1, cmap='rainbow', edgecolor='none')
    ax.grid()
    ax.set_axisbelow(True)
    ax.set_xlim((-30,30))
    ax.set_ylim((-10,10))
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("x' [mrad]")
    plt.show()

## Generate quadrupole scan data

### Calculate transfer matrices of Q-scan

In [ ]:
finv = np.linspace(f0inv, f1inv, NQ)
qmat = np.ones((NQ,2,2))
qmat[:,0,1] = 0.
qmat[:,1,0] = -finv
drift = np.eye(2)
drift[0,1] = L
tmat = drift @ qmat

### Generate Q-scan data

In [ ]:
Nbin = 60
Xmin = -30.0e-3
Xmax =  30.0e-3
qscan = np.empty((N, NQ, Nbin))
for i in range(NQ):
    print(i)
    p1 = tmat[i,:,:] @ np.swapaxes(particles, 1, 2)
    for j in range(N):
        h,xe = np.histogram(p1[j,0,:], Nbin, (Xmin,Xmax))
        qscan[j,i,:] = h

### Check Q-scan data

In [ ]:
for i in range(10):
    fig, ax = plt.subplots(figsize=(3,2))
    df = (f0inv-f1inv)/(NQ-1)
    ax.pcolormesh(np.linspace(Xmin,Xmax,Nbin+1)*1.e3, np.linspace(f1inv-df*0.5,f0inv+df*0.5,NQ+1), qscan[i,:,:], cmap='rainbow')
    #ax.set_xlim((-200,200))
    #ax.set_ylim((-20,20))
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("Q strength [a.u.]")
    plt.show()

## Prepare data for CNN

### Split data into training, test, and validation

In [ ]:
n_train, n_test, n_val = 6000, 2000, 2000
y_scale = np.array([1.0e5, 0.05, 0.01])
y_data = np.array([emittance, alpha, beta]).T * y_scale
x_train, y_train = qscan[:n_train,:,:], y_data[:n_train,:]
x_test, y_test = qscan[n_train:n_train+n_test,:,:], y_data[n_train:n_train+n_test,:]
x_val, y_val = qscan[-n_val:,:,:], y_data[-n_val:,:]

### Input image dimensions

In [ ]:
img_rows, img_cols = Nbin, NQ
input_shape = (img_rows, img_cols, 1)

### Reshape and scale input data

In [ ]:
x_train1 = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1).astype(np.float32)/M
x_test1 = x_test.reshape(x_test.shape[0], img_rows, img_cols, 1).astype(np.float32)/M
x_val1 = x_val.reshape(x_val.shape[0], img_rows, img_cols, 1).astype(np.float32)/M

## Training of CNN

### Import modules

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras import backend as K

batch_size = 128
epochs = 30

### Make CNN model

In [ ]:
# Make model
model = Sequential()
model.add(Input(input_shape))
model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(3))
model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae'])

### Check model

In [ ]:
model.summary()

### Fit the model to data (Training)

In [ ]:
history = model.fit(x_train1, y_train, batch_size=2*batch_size, epochs=epochs, verbose=1, validation_data=(x_test1, y_test))
score = model.evaluate(x_test1, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

### Evolution of loss function

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(history.history['loss'], label='Loss')
ax.semilogy(history.history['val_loss'], label='Val Loss')
ax.set_title('Loss')
ax.set_xlabel('Epoch')
ax.grid()
ax.legend()
plt.show()

## Check results

### Spot check

In [ ]:
predictions = model.predict(x_val1)
print('                emittance,      alpha,       beta')
for i in range(10):
    print('prediction  ', predictions[i]/y_scale)
    print('ground truth', y_val[i]/y_scale)
    print('error       ', (y_val[i]-predictions[i])/y_scale)
    print()

### Compare phase-space distribution

In [ ]:
for i in range(10):
    data = particles[n_train+n_test+i,:,:]
    eps, a, b = predictions[i]/y_scale
    g = (1+a**2)/b
    cov = eps * np.array([[b, -a],[-a, g]])
    pred = multivariate_normal.rvs(mean=np.zeros(2), cov=cov, size=M)
    z = gaussian_kde(data.T)(data.T)
    idx = z.argsort()
    z2 = z[idx]
    z = gaussian_kde(pred.T)(pred.T)
    idx2 = z.argsort()
    z3 = z[idx2]
    fig, ax = plt.subplots(1, 2, figsize=(6,3))
    ax[0].scatter(data[idx,0]*1.0e3, data[idx,1]*1.0e3, c=z2, s=1, cmap='rainbow', edgecolor='none')
    ax[0].grid()
    ax[0].set_axisbelow(True)
    ax[0].set_xlim((-30,30))
    ax[0].set_ylim((-10,10))
    ax[0].set_xlabel("x [mm]")
    ax[0].set_ylabel("x' [mrad]")
    ax[0].set_title("Ground Truth")
    ax[1].scatter(pred[idx2,0]*1.0e3, pred[idx2,1]*1.0e3, c=z3, s=1, cmap='rainbow', edgecolor='none')
    ax[1].grid()
    ax[1].set_axisbelow(True)
    ax[1].set_xlim((-30,30))
    ax[1].set_ylim((-10,10))
    ax[1].set_xlabel("x [mm]")
    ax[1].set_ylabel("x' [mrad]")
    ax[1].set_title("Prediction")
    plt.tight_layout()
    plt.show()

### Compare emittance and Twiss parameters

In [ ]:
pred = predictions/y_scale
pred_emittance = pred[:,0]
pred_alpha = pred[:,1]
pred_beta = pred[:,2]
truth = y_val/y_scale
truth_emittance = truth[:,0]
truth_alpha = truth[:,1]
truth_beta = truth[:,2]
# Emittance
xy = np.vstack([truth_emittance, pred_emittance])
z = gaussian_kde(xy)(xy)
idx = z.argsort()
z2 = z[idx]
fig, ax = plt.subplots(figsize=(3,3))
ax.scatter(truth_emittance[idx]*1.0e6, pred_emittance[idx]*1.0e6, c=z2, s=1, cmap='rainbow', edgecolor='none')
ax.grid()
ax.set_axisbelow(True)
ax.set_xlabel("Truth [$\mu$m rad]")
ax.set_ylabel("Prediction [$\mu$m rad]")
ax.set_title("Emittance")
ax.set_xlim((0.,None))
ax.set_ylim((0.,None))
plt.show()
# Alpha
xy = np.vstack([truth_alpha, pred_alpha])
z = gaussian_kde(xy)(xy)
idx = z.argsort()
z2 = z[idx]
fig, ax = plt.subplots(figsize=(3,3))
ax.scatter(truth_alpha[idx], pred_alpha[idx], c=z2, s=1, cmap='rainbow', edgecolor='none')
ax.grid()
ax.set_axisbelow(True)
ax.set_xlabel("Truth")
ax.set_ylabel("Prediction")
ax.set_title("Alpha")
plt.show()
# Beta
xy = np.vstack([truth_beta, pred_beta])
z = gaussian_kde(xy)(xy)
idx = z.argsort()
z2 = z[idx]
fig, ax = plt.subplots(figsize=(3,3))
ax.scatter(truth_beta[idx], pred_beta[idx], c=z2, s=1, cmap='rainbow', edgecolor='none')
ax.grid()
ax.set_axisbelow(True)
ax.set_xlabel("Truth [m]")
ax.set_ylabel("Prediction [m]")
ax.set_title("Beta")
ax.set_xlim((0.,None))
ax.set_ylim((0.,None))
plt.show()